```
U-Net recovers spatial detail through SKIP CONNECTIONS:
   Encoder saves feature maps → Decoder concatenates them
   Great for: small datasets, medical imaging, fine boundaries

DeepLabV3+ recovers spatial detail through ASPP + DECODER:
   ASPP captures context at multiple scales simultaneously
   Decoder refines boundaries using low-level features
   Great for: natural images, complex scenes, multi-class segmentation

Key insight: U-Net = many skip connections
             DeepLabV3+ = powerful feature extractor + one lightweight decoder

Normal 3×3 Conv (dilation=1):    Dilated 3×3 Conv (dilation=2):
filters adjacent pixels          filters with gaps (holes)

X . X . X                        X . . X . . X
. . . . .                        . . . . . . .
X . X . X            →           . . . . . . .
. . . . .                        X . . X . . X
X . X . X                        . . . . . . .
                                 . . . . . . .
Receptive field: 3×3             X . . X . . X
                                 Receptive field: 7×7

Same number of parameters, LARGER receptive field!
dilation=6  → sees a 13×13 area
dilation=12 → sees a 25×25 area
dilation=18 → sees a 37×37 area

ASPP runs all 4 dilation rates in PARALLEL → captures context at all scales at once
```

In [ ]:
!pip install segmentation-models-pytorch albumentations roboflow --quiet

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import albumentations as A
from albumentations import ToTensorV2
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
  print(f"   GPU   : {torch.cuda.get_device_name(0)}")
  print(f"   VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   smp   : {smp.__version__}")

In [ ]:
from roboflow import Roboflow

API_KEY = 'YOUR_API_KEY_HERE'
WORKSPACE = 'YOUR_WORKSPACE_HERE'
PROJECT_NAME = 'YOUR_PROJECT_NAME_HERE'
VERSION = 1

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
dataset = project.version(VERSION).download('png-mask-semantic')

DATASET_DIR = dataset.location
print(f"Dataset at: {DATASET_DIR}")

In [ ]:
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, 'train', 'images')
TRAIN_MASK_DIR = os.path.join(DATASET_DIR, 'train', 'masks')
VAL_IMG_DIR = os.path.join(DATASET_DIR, 'valid', 'images')
VAL_MASK_DIR = os.path.join(DATASET_DIR, 'valid', 'masks')

In [ ]:
if os.path.exists(DATASET_DIR):
    print(f"Contents of {DATASET_DIR}: {os.listdir(DATASET_DIR)}")

for split, img_dir, msk_dir in [
    ('train', TRAIN_IMG_DIR, TRAIN_MASK_DIR),
    ('val', VAL_IMG_DIR, VAL_MASK_DIR),
]:
    if os.path.exists(img_dir) and os.path.exists(msk_dir):
        n_imgs = len([f for f in os.listdir(img_dir) if not f.startswith('.')])
        n_masks = len([f for f in os.listdir(msk_dir) if not f.startswith('.')])
        print(f"   {split}/images → {n_imgs} | {split}/masks → {n_masks}")
    else:
        print(f"   [!] Error: Path not found for {split}. Check if {img_dir} exists.")

In [ ]:
path_to_check = TRAIN_IMG_DIR if os.path.exists(TRAIN_IMG_DIR) else os.path.join(DATASET_DIR, 'train')

try:
    sample_name = sorted([f for f in os.listdir(path_to_check) if not f.startswith('.')])[0]
    base_name = os.path.splitext(sample_name)[0]
    print(f"Selected sample: {sample_name}")
except IndexError:
    print(f"Error: No files found in {path_to_check}")
except FileNotFoundError:
    print(f"Error: Directory {path_to_check} still not found. Please check the 'Contents' output above.")

In [ ]:
img_path = os.path.join(path_to_check, sample_name)
mask_options = [f for f in os.listdir(path_to_check) if base_name in f and 'mask' in f.lower()]
mask_file = mask_options[0] if mask_options else sample_name.replace('.jpg', '.png')
mask_path = os.path.join(path_to_check, mask_file)

img = cv2.imread(img_path)
if img is not None:
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is not None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(img);                axes[0].set_title(f'Image  {img.shape}');  axes[0].axis('off')
        axes[1].imshow(mask, cmap='gray');  axes[1].set_title(f'Mask   {mask.shape}'); axes[1].axis('off')
        plt.suptitle('Sample Image-Mask Pair', fontsize=13)
        plt.tight_layout(); plt.show()

        print(f"Unique mask pixel values: {np.unique(mask)}")
    else:
        print(f"Error: Mask not found at {mask_path}")
else:
    print(f"Error: Image not found at {img_path}")

In [ ]:
IMG_SIZE = 512
BATCH_SIZE = 4
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Rotate(limit=15, p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

In [ ]:
class SegmentationDataset(Dataset):
  def __init__(self, img_dir, msk_dir, transform=None):
    self.img_dir = img_dir
    self.msk_dir = msk_dir
    self.transform = transform

    IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
    self.images = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith(IMAGE_EXTS) and '_mask' not in f
    ])

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    img_name = self.images[idx]
    image = cv2.imread(os.path.join(self.img_dir, img_name))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    base = os.path.splitext(img_name)[0]
    mask_name = base + '_mask.png'
    mask_path = os.path.join(self.msk_dir, mask_name)

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)

    mask = (mask > 0).astype(np.uint8)

    if self.transform:
      augmented = self.transform(image=image, mask=mask)
      image = augmented['image']
      mask = augmented['mask']

    if len(mask.shape) == 2:
        mask = mask.unsqueeze(0)

    return image, mask.float()

In [ ]:
TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VAL_DIR = os.path.join(DATASET_DIR, 'valid')

train_dataset = SegmentationDataset(TRAIN_DIR, TRAIN_DIR, train_transform)
val_dataset = SegmentationDataset(VAL_DIR, VAL_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Created loaders. Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

In [ ]:
train_dataset = SegmentationDataset(TRAIN_DIR, TRAIN_DIR, train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

imgs, msks = next(iter(train_loader))

print(f"\n Batch shapes:")
print(f"   images : {imgs.shape}")
print(f"   masks  : {msks.shape}")
print(f"   mask range: {msks.min():.0f} – {msks.max():.0f}")

# **Architecture Deep Dive: With Temporary Model**

In [ ]:
def show_receptive_field(dilation, ax, color):
  grid_size = 25
  center = grid_size // 2
  kernel = 3

  grid = np.zeros((grid_size, grid_size))

  offsets = [(-1, -1), (-1, 0), (-1, 1),
             (0, -1),  (0, 0),  (0, 1),
             (1, -1),  (1, 0),  (1, 1)]


  for dr, dc in offsets:
    r = center + dr * dilation
    c = center + dc * dilation

    if 0 <= r < grid_size and 0 <= c < grid_size:
      grid[r, c] = 1.0

  ax.imshow(grid, cmap='Blues', vmin=0, vmax=1)
  ax.plot(center, center, 'r*', markersize=12)
  effective_rf = 2 * dilation * (kernel - 1) + 1
  ax.set_title(f'Dilation = {dilation}\nReceptive Field: {effective_rf}×{effective_rf}', fontsize=10)
  ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
for ax, dilation, color in zip(axes, [1, 6, 12, 18], colors):
    show_receptive_field(dilation, ax, color)

plt.suptitle(
    'ASPP: 4 parallel dilated convolutions, each sees a different scale\n'
    '(Red star = center pixel being classified, Blue = sampled positions)',
    fontsize=12
)
plt.tight_layout()
plt.show()

print(" Key insight:")
print("   dilation=1  sees local texture  → good for thin boundaries")
print("   dilation=6  sees small objects  → captures local context")
print("   dilation=12 sees medium regions → captures object parts")
print("   dilation=18 sees large regions  → captures full object context")
print("   ASPP concatenates all 4 → the model knows context at ALL scales")

In [ ]:
temp_model = smp.DeepLabV3Plus(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1
)

print(" DeepLabV3+ internal components in smp:")
print()
print(f"  encoder         : {type(temp_model.encoder).__name__}")
print(f"  decoder         : {type(temp_model.decoder).__name__}")
print(f"  segmentation head: {type(temp_model.segmentation_head).__name__}")
print()

In [ ]:
decoder = temp_model.decoder
for name, module in decoder.named_children():
    print(f"    {name}: {type(module).__name__}")

del temp_model

---
## Build the DeepLabV3+ Model

### ResNet34 vs ResNet50 — When to Upgrade

```
ResNet34  (21M params):
   Faster training
   Less VRAM
   Enough for small/medium datasets (<5000 images)
   Start here

ResNet50  (25M params):
   Stronger feature extraction
   Better on complex multi-class scenes
   Needs more VRAM — reduce batch_size to 2 on T4 with 512×512
   Upgrade when ResNet34 has plateaued

EfficientNet-b4 (advanced, not covered here):
   State-of-art accuracy per parameter
   Much slower to converge
```

In [ ]:
ENCODER = 'resnet34'

model = smp.DeepLabV3Plus(
    encoder_name=ENCODER,
    encoder_weights='imagenet',
    in_channels=3,
    classes=1
).to(device)

In [ ]:
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
  out = model(dummy)

print(f" Model built: DeepLabV3+ ({ENCODER})")
print(f"   Input  : {dummy.shape}")
print(f"   Output : {out.shape}  same H×W as input")
print()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f" Total parameters     : {total_params:,}  ({total_params/1e6:.1f}M)")
print(f"   Trainable parameters : {trainable_params:,}")

In [ ]:
models_to_compare = {
    'DeepLabV3+ ResNet34' : smp.DeepLabV3Plus(encoder_name='resnet34', encoder_weights=None, classes=1),
    'DeepLabV3+ ResNet50' : smp.DeepLabV3Plus(encoder_name='resnet50', encoder_weights=None, classes=1),
    'U-Net ResNet34'      : smp.Unet(encoder_name='resnet34',           encoder_weights=None, classes=1),
    'U-Net++ ResNet34'    : smp.UnetPlusPlus(encoder_name='resnet34',   encoder_weights=None, classes=1),
}

print(" Model Comparison:")
print(f"{'Model':<25} {'Params':>12}")
print("-" * 40)
for name, m in models_to_compare.items():
    p = sum(x.numel() for x in m.parameters())
    print(f"{name:<25} {p:>10,}  ({p/1e6:.1f}M)")


In [ ]:
class DiceLoss(nn.Module):
  def __init__(self, smooth=1.0):
    super().__init__()
    self.smooth = smooth

  def forward(self, logits, targets):
    preds = torch.sigmoid(logits).view(logits.shape[0], -1)
    targets = targets.view(targets.shape[0], -1)

    intersection = (preds * targets).sum(dim=1)

    dice = (2.0 * intersection + self.smooth) / (
            preds.sum(dim=1) + targets.sum(dim=1) + self.smooth
        )
    return 1.0 - dice.mean()

In [ ]:
def compute_iou(logits, targets, threshold=0.5):
  with torch.no_grad():
    preds = (torch.sigmoid(logits) > threshold).float().view(logits.shape[0], -1)
    targets = targets.view(targets.shape[0], -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection

    iou = torch.where(union == 0, torch.ones_like(union), intersection / union)

  return iou.mean().item()

In [ ]:
bce_fn = nn.BCEWithLogitsLoss()
dice_fn = DiceLoss()

def combined_loss(logits, targets):
  return bce_fn(logits, targets) + dice_fn(logits, targets)

dummy_logits  = torch.randn(2, 1, IMG_SIZE, IMG_SIZE)
dummy_targets = torch.randint(0, 2, (2, 1, IMG_SIZE, IMG_SIZE)).float()
print(f"Loss test  : {combined_loss(dummy_logits, dummy_targets):.4f}")
print(f"IoU  test  : {compute_iou(dummy_logits, dummy_targets):.4f}")
print(" Loss and metric functions ready")

In [ ]:
optimizer = torch.optim.Adam([
    {'params': model.encoder.parameters(), 'lr': 1e-4},
    {'params': model.decoder.parameters(), 'lr': 3e-4},
    {'params': model.segmentation_head.parameters(), 'lr': 3e-4}
], weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
  model.train()
  total_loss , total_iou = 0.0, 0.0
  for imgs, msks in loader:
    imgs, msks = imgs.to(device), msks.to(device)
    optimizer.zero_grad()
    logits = model(imgs)
    loss = combined_loss(logits, msks)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    total_iou += compute_iou(logits, msks)
  n = len(loader)
  return total_loss / n, total_iou / n

In [ ]:
def validate(model, loader, device):
  model.eval()

  total_loss, total_iou = 0.0, 0.0
  with torch.no_grad():
    for imgs, msks in loader:
      imgs, msks = imgs.to(device), msks.to(device)
      logits = model(imgs)
      loss = combined_loss(logits, msks)

      total_loss += loss.item()
      total_iou += compute_iou(logits, msks)

  n = len(loader)
  return total_loss / n, total_iou / n

In [ ]:
NUM_EPOCHS   = 20
SAVE_PATH    = f'/content/deeplabv3plus_{ENCODER}_best.pth'
best_val_iou = 0.0
history      = {'train_loss': [], 'val_loss': [], 'train_iou': [], 'val_iou': []}

print(f"  Training DeepLabV3+ ({ENCODER})")
print("=" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
  train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, scheduler, device)
  val_loss, val_iou = validate(model, val_loader, device)

  scheduler.step(val_iou)

  if val_iou > best_val_iou:
    best_val_iou = val_iou
    torch.save(model.state_dict(), SAVE_PATH)
    marker = ' ← best saved'

  else:
    marker = ''

  history['train_loss'].append(train_loss)
  history['val_loss'].append(val_loss)
  history['train_iou'].append(train_iou)
  history['val_iou'].append(val_iou)

  lr_now = optimizer.param_groups[0]['lr']
  print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}  IoU: {train_iou:.4f} | "
        f"Val Loss: {val_loss:.4f}  IoU: {val_iou:.4f} | "
        f"LR: {lr_now:.2e}"
        f"{marker}"
    )

In [ ]:
epochs_x = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, history['train_loss'], 'b-o', markersize=4, label='Train Loss')
axes[0].plot(epochs_x, history['val_loss'],   'r-o', markersize=4, label='Val Loss')
axes[0].set_title('Loss (BCE + Dice)', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_x, history['train_iou'], 'b-o', markersize=4, label='Train IoU')
axes[1].plot(epochs_x, history['val_iou'],   'r-o', markersize=4, label='Val IoU')
axes[1].axhline(y=0.5, color='gray',  linestyle='--', alpha=0.5, label='IoU = 0.5')
axes[1].axhline(y=0.7, color='green', linestyle='--', alpha=0.5, label='IoU = 0.7 (good)')
axes[1].set_title('IoU (Intersection over Union)', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('IoU')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'DeepLabV3+ ({ENCODER}) Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Best Val IoU : {best_val_iou:.4f}")
print()
print(" Reading the curve:")
print("  Train↓ Val↓ together  → healthy training")
print("  Train↓ Val→ or Val↑   → overfitting (need more data or regularization)")
print("  Both still high        → underfitting (need more epochs or larger model)")

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()
print(f" Model loaded from: {SAVE_PATH}")

In [ ]:
IMAGENET_MEAN_NP = np.array([0.485, 0.456, 0.406])
IMAGENET_STD_NP  = np.array([0.229, 0.224, 0.225])

def denormalize(tensor):
  img = tensor.permute(1, 2, 0).numpy()
  return (img * IMAGENET_STD_NP + IMAGENET_MEAN_NP).clip(0, 1)

In [ ]:
def visualize_predictions(model, dataset, device, n_samples=4, threshold=0.5):
  model.eval()

  indices = np.random.choice(len(dataset), min(n_samples, len(dataset), replace=False))

  fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
  if n_samples == 1: axes = axes[np.newaxis, :]

  for col, title in enumerate(['Input Image', 'Ground Truth', f'Prediction (t={threshold})']):
      axes[0, col].set_title(title, fontsize=11, fontweight='bold')

  for row, idx in enumerate(indices):
      image, gt_mask = dataset[idx]

      with torch.no_grad():
          logit = model(image.unsqueeze(0).to(device))
          prob  = torch.sigmoid(logit).squeeze().cpu().numpy()
          pred  = (prob > threshold).astype(np.uint8)

      img_display = denormalize(image)
      gt_display  = gt_mask.squeeze().numpy()

      inter = (pred * gt_display).sum()
      union = (pred + gt_display).clip(0, 1).sum()
      iou   = inter / union if union > 0 else 1.0

      axes[row, 0].imshow(img_display); axes[row, 0].axis('off')
      axes[row, 1].imshow(gt_display, cmap='Blues'); axes[row, 1].axis('off')
      axes[row, 2].imshow(pred, cmap='Reds')
      axes[row, 2].set_xlabel(f'IoU = {iou:.3f}', fontsize=10)
      axes[row, 2].axis('off')

  plt.suptitle(f'DeepLabV3+ ({ENCODER}) Inference', fontsize=13)
  plt.tight_layout()
  plt.show()

In [ ]:
image, gt_mask = val_dataset[0]

with torch.no_grad():
  logit = model(image.unsqueeze(0).to(device))
  prob = torch.sigmoid(logit).squeeze().cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(denormalize(image))
axes[0].set_title('Image', fontsize=11); axes[0].axis('off')

axes[1].imshow(gt_mask.squeeze().numpy(), cmap='Blues')
axes[1].set_title('Ground Truth', fontsize=11); axes[1].axis('off')

im = axes[2].imshow(prob, cmap='hot', vmin=0, vmax=1)
axes[2].set_title('Probability Map\n(bright = confident foreground)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

axes[3].imshow(prob > 0.5, cmap='Reds')
axes[3].set_title('Binary Prediction\n(threshold = 0.5)', fontsize=11)
axes[3].axis('off')

plt.suptitle('DeepLabV3+ — Probability Heatmap Detail', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files

size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f"Model size: {size_mb:.1f} MB")
files.download(SAVE_PATH)

```
 How to reload later ──────────────────────────────────────────────────────

 model_reload = smp.DeepLabV3Plus(
     encoder_name='resnet34',   # must match what you trained
     encoder_weights=None,      # don't reload imagenet weights — we load our own
     in_channels=3,
     classes=1
 )
 model_reload.load_state_dict(
     torch.load('deeplabv3plus_resnet34_best.pth', map_location=device)
 )
 model_reload = model_reload.to(device)
 model_reload.eval()
 print('Model reloaded')
```